# 06.3 Logical Operators and Short-Circuiting

Three keywords — `and`, `or`, `not` — with one surprising property: **`and` and
`or` do not return booleans.** They return one of their operands.

That single fact explains the `or`-default idiom, the `and`-guard idiom, and a
whole family of bugs.

**40 numbered examples.**

## Theory

### What they actually return

```python
a and b     # if a is falsy, return a; otherwise return b
a or b      # if a is truthy, return a; otherwise return b
not a       # always a real bool
```

Read those rules carefully. `and` returns the **first falsy** operand or the
last one. `or` returns the **first truthy** operand or the last one.

`not` is the exception — it always produces `True` or `False`.

### Short-circuiting

The second operand is evaluated **only when needed**:

```python
False and expensive()    # expensive() never runs
True  or  expensive()    # expensive() never runs
```

This is not an optimisation detail — it is guaranteed behaviour you can rely on,
and it is how guard patterns work:

```python
if user is not None and user.is_active:
    ...
```

If `user` is `None`, the second half never runs, so no `AttributeError`.

### Truthiness drives everything

These operators work on **truthiness** (05.5), not on booleans. Any object can
be an operand, and the falsy list is short: `False`, `None`, zero of any numeric
type, and empty collections.

### Precedence

```
not  >  and  >  or
```

So `not a or b` means `(not a) or b`, and `a or b and c` means `a or (b and c)`.
Bracket when it is not instantly obvious.

In [ ]:
# EXAMPLE 1-6: the truth tables.
print("EXAMPLE 1-6: truth tables")
print("")

print("   1-2. `and` - True only when both are true")
print("        A      B      A and B")
print("        " + "-" * 30)
for left in [True, False]:
    for right in [True, False]:
        print(f"        {str(left):<6} {str(right):<6} {left and right}")

print("")
print("   3-4. `or` - True when either is true")
print("        A      B      A or B")
print("        " + "-" * 30)
for left in [True, False]:
    for right in [True, False]:
        print(f"        {str(left):<6} {str(right):<6} {left or right}")

print("")
print("   5-6. `not` - inverts, and always returns a real bool")
for value in [True, False]:
    print(f"        not {str(value):<6} -> {not value}")

In [ ]:
# EXAMPLE 7-14: and/or return OPERANDS, not booleans.
print("EXAMPLE 7-14: they return operands")
print("")

print("   `or` returns the first TRUTHY operand, else the last:")
print("   7.  'a' or 'b'      ->", repr("a" or "b"))
print("   8.  '' or 'b'       ->", repr("" or "b"))
print("   9.  0 or []         ->", repr(0 or []), "<- both falsy, last wins")
print("   10. None or 0 or 'x'->", repr(None or 0 or "x"))

print("")
print("   `and` returns the first FALSY operand, else the last:")
print("   11. 'a' and 'b'     ->", repr("a" and "b"))
print("   12. '' and 'b'      ->", repr("" and "b"))
print("   13. 1 and 2 and 3   ->", repr(1 and 2 and 3))
print("   14. 1 and 0 and 3   ->", repr(1 and 0 and 3), "<- stops at 0")

print("")
print("   Only `not` guarantees a bool:", type(not "a").__name__)

In [ ]:
# EXAMPLE 15-20: short-circuiting, demonstrated.
print("EXAMPLE 15-20: short-circuiting")
print("")

call_log = []


def track(label, value):
    """Return a value, recording that it was evaluated."""
    call_log.append(label)
    return value


# 15. `and` stops at the first falsy operand.
call_log.clear()
result = track("first", False) and track("second", True)
print("   15. False and True")
print("       result:", result, " evaluated:", call_log)

# 16. `or` stops at the first truthy operand.
call_log.clear()
result = track("first", True) or track("second", False)
print("")
print("   16. True or False")
print("       result:", result, " evaluated:", call_log)

# 17. Both evaluated when the first does not decide it.
call_log.clear()
result = track("first", True) and track("second", False)
print("")
print("   17. True and False")
print("       result:", result, " evaluated:", call_log)

# 18. Chains stop as soon as the answer is known.
call_log.clear()
result = (track("a", False) and track("b", True) and track("c", True))
print("")
print("   18. False and True and True")
print("       result:", result, " evaluated:", call_log)

# 19-20. This is a guarantee, not an optimisation.
print("")
print("   19. You can RELY on the second operand not running.")
print("   20. That is what makes guard patterns safe.")

## The idioms these enable

In [ ]:
# EXAMPLE 21-26: guard patterns.
print("EXAMPLE 21-26: guarding with `and`")
print("")


class User:
    """A user who may or may not be active."""

    def __init__(self, name, active):
        self.name = name
        self.active = active


# 21. Guarding an attribute access.
for user in [User("Asha", True), User("Ben", False), None]:
    # If user is None, the second half never runs.
    can_post = user is not None and user.active
    label = user.name if user else "None"
    print(f"   21. user={label:<6} can_post={can_post}")

# 22. Without the guard, it crashes.
print("")
user = None
try:
    user.active
except AttributeError as error:
    print("   22. without the guard:", error)

# 23. Guarding a division.
print("")
for count in [4, 0]:
    average = count > 0 and 100 / count
    print(f"   23. count={count} -> {average}")

# 24. Guarding an index.
items = [10, 20]
for index in [0, 5]:
    value = index < len(items) and items[index]
    print(f"   24. index={index} -> {value}")

# 25. Guarding a dict lookup.
config = {"retries": 3}
for key in ["retries", "timeout"]:
    value = key in config and config[key]
    print(f"   25. key={key!r:<10} -> {value}")

# 26. The clearer alternative for dicts.
print("")
print("   26. For dicts, .get() reads better:", config.get("timeout", "default"))

In [ ]:
# EXAMPLE 27-32: the `or` default idiom, and its trap.
print("EXAMPLE 27-32: defaults with `or`")
print("")

# 27. The common idiom.
def greet(name=None):
    """Greet by name, falling back to Anonymous."""
    display = name or "Anonymous"
    return f"Hello, {display}"


print("   27. greet('Asha') ->", greet("Asha"))
print("       greet(None)   ->", greet(None))
print("       greet('')     ->", greet(""))

# 28. THE TRAP: it also replaces valid falsy values.
def set_limit(limit=None):
    """Apply a default limit - broken for 0."""
    return limit or 10


print("")
print("   28. set_limit(5) ->", set_limit(5))
print("       set_limit(0) ->", set_limit(0), "<- WRONG, 0 was intended")

# 29. The correct version.
def set_limit_fixed(limit=None):
    """Apply a default only when limit is genuinely absent."""
    return 10 if limit is None else limit


print("")
print("   29. fixed: set_limit_fixed(0) ->", set_limit_fixed(0))

# 30-32. When `or` IS appropriate.
print("")
print("   30. `or` is fine when ALL falsy values should be replaced:")
print("       user_input = '' -> 'default' is correct here")

print("")
print("   31. Chained fallbacks read well:")
primary, secondary, fallback = None, "", "default"
print("       primary or secondary or fallback ->",
      repr(primary or secondary or fallback))

print("")
print("   32. Rule: use `or` for 'any falsy means missing'.")
print("       Use `is None` when 0 or '' are real values.")

## Precedence and common mistakes

In [ ]:
# EXAMPLE 33-36: precedence.
print("EXAMPLE 33-36: not > and > or")
print("")

a, b, c = True, False, True

# 33. not binds tightest.
print("   33. not a or b")
print("       means (not a) or b ->", (not a) or b)
print("       NOT not (a or b)   ->", not (a or b))

# 34. and binds tighter than or.
print("")
print("   34. a or b and c")
print("       means a or (b and c) ->", a or (b and c))
print("       NOT (a or b) and c   ->", (a or b) and c)

# 35. Comparison binds tighter than all of them.
value = 5
print("")
print("   35. value > 3 and value < 10")
print("       means (value > 3) and (value < 10) ->",
      (value > 3) and (value < 10))
print("       better written as a chain: 3 < value < 10 ->", 3 < value < 10)

# 36. Arithmetic binds tighter still.
print("")
print("   36. 2 + 3 > 4 and 1 < 2")
print("       means ((2+3) > 4) and (1 < 2) ->", ((2 + 3) > 4) and (1 < 2))

In [ ]:
# EXAMPLE 37-40: mistakes people actually make.
print("EXAMPLE 37-40: common mistakes")
print("")

# 37. Comparing against several values.
value = 5
print("   37. WRONG: value == 1 or 2 or 3")
print("       ->", value == 1 or 2 or 3, "<- this is truthy for ANY value")
print("       because it parses as (value == 1) or 2 or 3")
print("       RIGHT: value in (1, 2, 3) ->", value in (1, 2, 3))

# 38. Using & instead of and.
print("")
print("   38. `and` vs `&`:")
print("       True and False ->", True and False)
print("       True & False   ->", True & False, "<- bitwise, works by accident")
print("       1 and 2        ->", 1 and 2)
print("       1 & 2          ->", 1 & 2, "<- completely different")
print("       & does NOT short-circuit. Use `and` for logic.")

# 39. Assuming a bool comes back.
print("")
print("   39. the return value is an operand, not a bool:")
result = [] or "fallback"
print("       [] or 'fallback' ->", repr(result), type(result).__name__)
print("       wrap in bool() if you genuinely need True/False:",
      bool([] or "fallback"))

# 40. not with `in` and `is`.
print("")
print("   40. `not in` and `is not` are single operators:")
print("       3 not in [1, 2]   ->", 3 not in [1, 2])
print("       not (3 in [1, 2]) ->", not (3 in [1, 2]), "<- same, less readable")
print("       None is not False ->", None is not False)

## Takeaways

1. `and` and `or` return **operands**, not booleans. Only `not` guarantees a
   `bool`.
2. `or` returns the first **truthy** operand; `and` returns the first **falsy**
   one. Either falls back to the last operand.
3. **Short-circuiting is guaranteed** — you can rely on the second operand not
   being evaluated.
4. That guarantee makes guard patterns safe:
   `user is not None and user.active`.
5. The `x or default` idiom replaces **every** falsy value — a bug when `0` or
   `""` is legitimate. Use `is None` there.
6. Precedence is `not` > `and` > `or`, and all three bind looser than
   comparisons.
7. `value == 1 or 2 or 3` is always truthy. Use `value in (1, 2, 3)`.
8. `&` is bitwise and does **not** short-circuit — never use it for logic.

## Try it yourself

1. Predict the value and type of: `[] or 0`, `1 and []`, `not []`.
2. Write a function that crashes without a guard, then add one.
3. Show `x or 10` failing for `x = 0`, then fix it.
4. Explain why `5 == 1 or 2` is `2` rather than `False`.
5. Write a chain of three fallbacks using `or`.